<a href="https://colab.research.google.com/github/positivefunctionIN/Medical_Imaging_using_CNN/blob/main/Model3_ResNet50.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install kaggle -q

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import cv2
import os
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import seaborn as sns
from google.colab import files
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, Flatten, Dense, Dropout,
    GlobalAveragePooling2D, GlobalMaxPooling2D,
    Add, Multiply, Concatenate, Activation, Lambda
)
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

print("TensorFlow version:", tf.__version__)

# Kaggle API
import os
os.environ["KAGGLE_API_TOKEN"] = "KGAT_5b435f7615b49cc51191f5ab984c36d2"

!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia
!unzip -q chest-xray-pneumonia.zip -d chest_xray_data

data_path = "/content/chest_xray_data/chest_xray"

# Set constants
IMG_SIZE = 224
BATCH_SIZE = 32
CLASS_NAMES = ['Normal', 'Pneumonia']
NUM_CLASSES = 1  # Binary classification

TensorFlow version: 2.20.0
Dataset URL: https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia
License(s): other
100% 2.29G/2.29G [00:20<00:00, 122MB/s] 



In [2]:
# Data augmentation for training
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    validation_split=0.2
)

test_datagen = ImageDataGenerator(rescale=1./255)

# Training generator (80% of train data)
train_generator = train_datagen.flow_from_directory(
    f"{data_path}/train",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='training',
    shuffle=True
)

# Validation generator (20% of train data)
val_generator = train_datagen.flow_from_directory(
    f"{data_path}/train",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='validation',
    shuffle=False
)

# Test generator
test_generator = test_datagen.flow_from_directory(
    f"{data_path}/test",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

print(f"\n✅ Train samples: {train_generator.samples}")
print(f"✅ Val samples: {val_generator.samples}")
print(f"✅ Test samples: {test_generator.samples}")
print(f"✅ Class mapping: {train_generator.class_indices}")

Found 4173 images belonging to 2 classes.
Found 1043 images belonging to 2 classes.
Found 624 images belonging to 2 classes.

✅ Train samples: 4173
✅ Val samples: 1043
✅ Test samples: 624
✅ Class mapping: {'NORMAL': 0, 'PNEUMONIA': 1}


In [3]:
print("\n" + "="*60)
print("🔵 MODEL 1: Custom CNN ")
print("="*60)

def build_custom_cnn():
    model = Sequential([
        Conv2D(32, (3, 3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 3)),
        MaxPooling2D(2, 2),

        Conv2D(64, (3, 3), activation='relu'),
        MaxPooling2D(2, 2),

        Conv2D(128, (3, 3), activation='relu'),
        MaxPooling2D(2, 2),

        Conv2D(256, (3, 3), activation='relu'),
        MaxPooling2D(2, 2),

        Flatten(),
        Dense(512, activation='relu'),
        Dropout(0.5),
        Dense(NUM_CLASSES, activation='sigmoid')
    ])
    return model

custom_cnn = build_custom_cnn()
custom_cnn.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("✅ Custom CNN built")
custom_cnn.summary()

# Train
custom_history = custom_cnn.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10,
    callbacks=[EarlyStopping(patience=3, restore_best_weights=True)],
    verbose=1
)

# Evaluate
custom_test_loss, custom_test_acc = custom_cnn.evaluate(test_generator, verbose=0)
print(f"\n📊 Custom CNN Test Accuracy: {custom_test_acc:.2%}")


🔵 MODEL 1: Custom CNN 


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


✅ Custom CNN built


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 24, 24, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 12, 12, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 36864)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │    18,874,880 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           513 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,263,809 (73.49 MB)

 Trainable params: 19,263,809 (73.49 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 127s 880ms/step - accuracy: 0.7877 - loss: 0.4774 - val_accuracy: 0.8447 - val_loss: 0.3270
Epoch 2/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 109s 836ms/step - accuracy: 0.8852 - loss: 0.2704 - val_accuracy: 0.8945 - val_loss: 0.2567
Epoch 3/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 106s 805ms/step - accuracy: 0.9061 - loss: 0.2303 - val_accuracy: 0.8782 - val_loss: 0.2645
Epoch 4/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 103s 788ms/step - accuracy: 0.9197 - loss: 0.2118 - val_accuracy: 0.9262 - val_loss: 0.1778
Epoch 5/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 101s 776ms/step - accuracy: 0.9262 - loss: 0.1962 - val_accuracy: 0.9204 - val_loss: 0.1930
Epoch 6/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 101s 775ms/step - accuracy: 0.9336 - loss: 0.1749 - val_accuracy: 0.9252 - val_loss: 0.1738
Epoch 7/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 101s 772ms/step - accuracy: 0.9370 - loss: 0.1688 - val_accuracy: 0.9406 - val_loss: 0.1518
Epoch 8/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 104s 794ms/step - accuracy: 0.9353 -

In [4]:
def channel_attention(input_feature, ratio=16):
    channel = input_feature.shape[-1]
    avg_pool = GlobalAveragePooling2D()(input_feature)
    max_pool = GlobalMaxPooling2D()(input_feature)
    dense1 = Dense(channel // ratio, activation='relu')
    dense2 = Dense(channel, activation='sigmoid')
    avg_out = dense2(dense1(avg_pool))
    max_out = dense2(dense1(max_pool))
    attention = Add()([avg_out, max_out])
    attention = Activation('sigmoid')(attention)
    attention = tf.expand_dims(tf.expand_dims(attention, 1), 1)
    return Multiply()([input_feature, attention])

def spatial_attention(input_feature):
    avg_pool = tf.reduce_mean(input_feature, axis=-1, keepdims=True)
    max_pool = tf.reduce_max(input_feature, axis=-1, keepdims=True)
    concat = Concatenate(axis=-1)([avg_pool, max_pool])
    attention = Conv2D(1, kernel_size=7, padding='same', activation='sigmoid')(concat)
    return Multiply()([input_feature, attention])

def cbam_block(input_feature, ratio=16):
    x = channel_attention(input_feature, ratio)
    x = spatial_attention(x)
    return x

In [9]:
import tensorflow as tf
from tensorflow.keras.layers import Layer, Conv2D, Dense, GlobalAveragePooling2D, GlobalMaxPooling2D, Add, Multiply, Concatenate, Activation, MaxPooling2D, Flatten, Dropout
from tensorflow.keras.models import Sequential

class CBAMBlock(Layer):
    """
    Convolutional Block Attention Module (CBAM) as a custom layer.
    """
    def __init__(self, ratio=16, **kwargs):
        super(CBAMBlock, self).__init__(**kwargs)
        self.ratio = ratio

        self.global_avg_pool = GlobalAveragePooling2D()
        self.global_max_pool = GlobalMaxPooling2D()
        self.dense1 = Dense(1, activation='relu')
        self.dense2 = Dense(1, activation='sigmoid')

        self.concat = Concatenate(axis=-1)
        self.conv = Conv2D(1, kernel_size=7, padding='same', activation='sigmoid')

        self.multiply_channel = Multiply()
        self.multiply_spatial = Multiply()

    def build(self, input_shape):

        channels = input_shape[-1]
        self.dense1 = Dense(channels // self.ratio, activation='relu')
        self.dense2 = Dense(channels, activation='sigmoid')
        super(CBAMBlock, self).build(input_shape)

    def call(self, inputs):
        avg_pool = self.global_avg_pool(inputs)
        max_pool = self.global_max_pool(inputs)

        avg_out = self.dense2(self.dense1(avg_pool))
        max_out = self.dense2(self.dense1(max_pool))

        channel_attention = Add()([avg_out, max_out])
        channel_attention = Activation('sigmoid')(channel_attention)

        channel_attention = tf.expand_dims(tf.expand_dims(channel_attention, axis=1), axis=1)

        refined_inputs = self.multiply_channel([inputs, channel_attention])


        avg_pool_spatial = tf.reduce_mean(refined_inputs, axis=-1, keepdims=True)
        max_pool_spatial = tf.reduce_max(refined_inputs, axis=-1, keepdims=True)

        spatial_concat = self.concat([avg_pool_spatial, max_pool_spatial])
        spatial_attention = self.conv(spatial_concat)

        refined_output = self.multiply_spatial([refined_inputs, spatial_attention])

        return refined_output

def build_hybrid_cnn():
    model = Sequential([
        Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 3)),
        MaxPooling2D(2, 2),
        CBAMBlock(ratio=16),
        Conv2D(64, (3, 3), activation='relu'),
        MaxPooling2D(2, 2),
        CBAMBlock(ratio=8),

        Conv2D(128, (3, 3), activation='relu'),
        MaxPooling2D(2, 2),
        CBAMBlock(ratio=4),

        Conv2D(256, (3, 3), activation='relu'),
        MaxPooling2D(2, 2),

        Flatten(),
        Dense(512, activation='relu'),
        Dropout(0.5),
        Dense(1, activation='sigmoid')
    ])
    return model


hybrid_cnn = build_hybrid_cnn()
hybrid_cnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

hybrid_history = hybrid_cnn.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10,
    callbacks=[EarlyStopping(patience=3, restore_best_weights=True)],
    verbose=1
)

Epoch 1/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 120s 832ms/step - accuracy: 0.7359 - loss: 0.5570 - val_accuracy: 0.7411 - val_loss: 0.4428
Epoch 2/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 102s 779ms/step - accuracy: 0.8399 - loss: 0.3583 - val_accuracy: 0.8706 - val_loss: 0.2870
Epoch 3/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 101s 774ms/step - accuracy: 0.8924 - loss: 0.2639 - val_accuracy: 0.8667 - val_loss: 0.2847
Epoch 4/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 107s 814ms/step - accuracy: 0.8974 - loss: 0.2452 - val_accuracy: 0.9022 - val_loss: 0.2074
Epoch 5/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 102s 781ms/step - accuracy: 0.9006 - loss: 0.2215 - val_accuracy: 0.9128 - val_loss: 0.2014
Epoch 6/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 101s 772ms/step - accuracy: 0.9135 - loss: 0.2152 - val_accuracy: 0.9012 - val_loss: 0.2165
Epoch 7/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 100s 765ms/step - accuracy: 0.9281 - loss: 0.1946 - val_accuracy: 0.9233 - val_loss: 0.1707
Epoch 8/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 101s 772ms/step - accuracy: 0.9209 -

In [10]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import GlobalAveragePooling2D

def build_resnet50():
    base = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    base.trainable = False  # Freeze base

    model = Sequential([
        base,
        GlobalAveragePooling2D(),
        Dense(512, activation='relu'),
        Dropout(0.5),
        Dense(1, activation='sigmoid')
    ])
    return model

resnet50 = build_resnet50()
resnet50.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [16]:
print("🔴 TRAINING RESNET50")

from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

resnet_history = resnet50.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10,
    callbacks=[EarlyStopping(patience=3, restore_best_weights=True)],
    verbose=1
)
resnet_test_loss, resnet_test_acc = resnet50.evaluate(test_generator, verbose=0)
print(f"\n📊 ResNet50 Test Accuracy (frozen): {resnet_test_acc:.4f}")

print("\n🔥 Fine-tuning ResNet50...")
base = resnet50.layers[0]
base.trainable = True

resnet50.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

resnet_finetune_history = resnet50.fit(
    train_generator,
    validation_data=val_generator,
    epochs=5,
    callbacks=[EarlyStopping(patience=3, restore_best_weights=True)],
    verbose=1
)

resnet_test_loss, resnet_test_acc = resnet50.evaluate(test_generator, verbose=0)
print(f"\n📊 ResNet50 Test Accuracy (fine-tuned): {resnet_test_acc:.4f}")

class CombinedHistory:
    def __init__(self, history1, history2):
        self.history = {}
        for key in history1.history.keys():
            self.history[key] = history1.history[key] + history2.history[key]

resnet_combined_history = CombinedHistory(resnet_history, resnet_finetune_history)

🔴 TRAINING RESNET50
Epoch 1/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 123s 859ms/step - accuracy: 0.7254 - loss: 0.5841 - val_accuracy: 0.7440 - val_loss: 0.5045
Epoch 2/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 102s 782ms/step - accuracy: 0.7551 - loss: 0.5025 - val_accuracy: 0.7555 - val_loss: 0.4664
Epoch 3/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 103s 789ms/step - accuracy: 0.7726 - loss: 0.4520 - val_accuracy: 0.7670 - val_loss: 0.4362
Epoch 4/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 103s 790ms/step - accuracy: 0.7906 - loss: 0.4270 - val_accuracy: 0.7699 - val_loss: 0.4269
Epoch 5/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 102s 778ms/step - accuracy: 0.7934 - loss: 0.4301 - val_accuracy: 0.8063 - val_loss: 0.4095
Epoch 6/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 103s 786ms/step - accuracy: 0.7987 - loss: 0.4182 - val_accuracy: 0.7872 - val_loss: 0.4078
Epoch 7/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 102s 776ms/step - accuracy: 0.8069 - loss: 0.4009 - val_accuracy: 0.7776 - val_loss: 0.4532
Epoch 8/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 102s 779ms/step 